In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report

# ========== 1. 讀檔 ==========
winner = pd.read_csv('./data/winners.csv')
drivers = pd.read_csv('./data/drivers_updated.csv')
teams = pd.read_csv('./data/teams_updated.csv')

# ========== 2. 新增 year 欄位 ==========
winner['year'] = pd.to_datetime(winner['Date']).dt.year
winner['year_raw'] = winner['year']
winner['Grand Prix raw'] = winner['Grand Prix']

# ========== 3. 合併 drivers ==========
df = winner.merge(
    drivers[['Driver', 'Car', 'year', 'Nationality']],
    left_on=['Winner', 'Car', 'year'],
    right_on=['Driver', 'Car', 'year'],
    how='left'
)

# ========== 4. 合併 teams ==========
df = df.merge(
    teams[['Team', 'year']],
    left_on=['Car', 'year'],
    right_on=['Team', 'year'],
    how='left'
)

# ========== 5. 保留賽前可知特徵 ==========
# Winner 只作為 y，不作為特徵
cat_cols = ['Car', 'Grand Prix', 'Nationality', 'Team']  # 不能有 Winner!
num_cols = ['Laps', 'year']

# 填補數值缺漏
df[num_cols] = df[num_cols].fillna(0)

# ========== 6. 過濾只出現一次的冠軍 ==========
value_counts = df['Winner'].value_counts()
valid_drivers = value_counts[value_counts >= 2].index
df = df[df['Winner'].isin(valid_drivers)]

# ========== 7. 編碼 ==========
# 1. Winner 做為 y
le_winner = LabelEncoder()
df['Winner'] = le_winner.fit_transform(df['Winner'].astype(str))
y = df['Winner'].values

# 2. 其他欄位編碼
encoders = {}
for col in cat_cols:
    le2 = LabelEncoder()
    df[col] = le2.fit_transform(df[col].astype(str))
    encoders[col] = le2

# 數值標準化
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

# ========== 8. 準備特徵 ==========
X_cat = df[cat_cols].values
X_num = df[num_cols].values
num_classes = len(np.unique(y))

df = df.reset_index(drop=True)
df['orig_index'] = df.index

# ========== 9. train_test_split ==========
X_cat_train, X_cat_test, X_num_train, X_num_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_cat, X_num, y, df['orig_index'].values, test_size=0.2, random_state=42, stratify=y
)

# ========== 10. PyTorch Dataset ==========
class F1RaceSet(Dataset):
    def __init__(self, X_cat, X_num, y):
        self.X_cat = torch.tensor(X_cat, dtype=torch.long)
        self.X_num = torch.tensor(X_num, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X_cat[idx], self.X_num[idx], self.y[idx]

# ========== 11. Model ==========
class F1DNN(nn.Module):
    def __init__(self, cat_dims, num_num_features, embedding_dim=8, hidden_dim=128, num_classes=None):
        super().__init__()
        self.emb_layers = nn.ModuleList([
            nn.Embedding(cat_dim, embedding_dim) for cat_dim in cat_dims
        ])
        input_dim = embedding_dim * len(cat_dims) + num_num_features
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, num_classes)
        )
    def forward(self, x_cat, x_num):
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.emb_layers)]
        x = torch.cat(embs + [x_num], dim=1)
        return self.mlp(x)

batch_size = 128
trainset = F1RaceSet(X_cat_train, X_num_train, y_train)
testset = F1RaceSet(X_cat_test, X_num_test, y_test)
train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(testset, batch_size=batch_size)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cat_dims = [int(df[col].max() + 1) for col in cat_cols]

model = F1DNN(cat_dims, len(num_cols), embedding_dim=8, hidden_dim=128, num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 30

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_cat_batch, X_num_batch, y_batch in train_loader:
        X_cat_batch, X_num_batch, y_batch = X_cat_batch.to(device), X_num_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        output = model(X_cat_batch, X_num_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
    avg_loss = total_loss / len(trainset)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

# ========== 12. 評估 ==========
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for X_cat_batch, X_num_batch, y_batch in test_loader:
        X_cat_batch, X_num_batch = X_cat_batch.to(device), X_num_batch.to(device)
        logits = model(X_cat_batch, X_num_batch)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(y_batch.numpy())
all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

unique_y = np.unique(all_labels)
target_names = [f"{idx}: {name}" for idx, name in zip(unique_y, le_winner.inverse_transform(unique_y))]
print(classification_report(
    all_labels, all_preds,
    labels=unique_y,
    target_names=target_names,
    zero_division=0
))

torch.save(model.state_dict(), 'f1_dnn_embedding.pth')

# ========== 13. 隨機抽10筆測試集預測結果 ==========
winner_decoder = le_winner
test_df = df.iloc[idx_test].reset_index(drop=True)
np.random.seed(42)
rand_idx = np.random.choice(len(test_df), size=50, replace=False)
for i in rand_idx:
    row = test_df.iloc[i]
    year = int(row['year_raw'])
    grand_prix = row['Grand Prix raw']
    true_winner = winner_decoder.inverse_transform([all_labels[i]])[0]
    pred_winner = winner_decoder.inverse_transform([all_preds[i]])[0]
    print(f"{year} {grand_prix}\nPredicted Winner: {pred_winner} , True Winner: {true_winner}\n")


Epoch 1/30, Loss: 4.3475
Epoch 2/30, Loss: 3.7768
Epoch 3/30, Loss: 3.3875
Epoch 4/30, Loss: 3.0324
Epoch 5/30, Loss: 2.7527
Epoch 6/30, Loss: 2.5070
Epoch 7/30, Loss: 2.3128
Epoch 8/30, Loss: 2.1144
Epoch 9/30, Loss: 1.9685
Epoch 10/30, Loss: 1.8300
Epoch 11/30, Loss: 1.6890
Epoch 12/30, Loss: 1.5954
Epoch 13/30, Loss: 1.4702
Epoch 14/30, Loss: 1.3791
Epoch 15/30, Loss: 1.2978
Epoch 16/30, Loss: 1.2071
Epoch 17/30, Loss: 1.1591
Epoch 18/30, Loss: 1.0800
Epoch 19/30, Loss: 1.0187
Epoch 20/30, Loss: 0.9908
Epoch 21/30, Loss: 0.9296
Epoch 22/30, Loss: 0.8845
Epoch 23/30, Loss: 0.8200
Epoch 24/30, Loss: 0.7954
Epoch 25/30, Loss: 0.7388
Epoch 26/30, Loss: 0.7268
Epoch 27/30, Loss: 0.6980
Epoch 28/30, Loss: 0.6325
Epoch 29/30, Loss: 0.6220
Epoch 30/30, Loss: 0.6129
                             precision    recall  f1-score   support

           0: Alain  Prost        0.77      1.00      0.87        10
             1: Alan Jones        0.40      1.00      0.57         2
        2: Alberto  A